In [ ]:
import re

In [ ]:
import zipfile
import os
from google.colab import files

In [ ]:
import pandas as pd

In [ ]:
from transformers import pipeline

# 1.Dataset in zip --> folder with txt files

Upload the zip folder with the txt files (the dataset) to the colab and save them into a list so that they can be looped over and each of them is separated into evaluations

result: directory with all txt files -- one txt=evaluations of one semester

In [ ]:
def prepare_colab_directory():
    # Upload a zip-directory from the computer
    uploaded = files.upload()

    # Get the name of the uploaded zip-dir.
    zip_file_name = list(uploaded.keys())[0]

    # Unzip the uploaded file
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall('unzipped_folder1')

    # Get a list of .txt files in the directory
    txt_files = []
    for filename in os.listdir('unzipped_folder1'):
        file_path = os.path.join('unzipped_folder1', filename)
        txt_files.append(file_path)  # Add the file path to the list of files

    return txt_files

#2.Separate into evaluations

Save the data set in the dictionary format

result: {'Winter_2023_en_S1f_1': ["The last block was a great .....

values are whole evaluations

In [ ]:
def separate_into_evaluations(input_file):
    evaluations = {}
    key, reaktionsblatt, body = None, None, ''

    with open(input_file, 'r', encoding='utf-8') as file:
        for line in file:
            # Search for first key-pattern: S[0-9]+[wfm]
            key_match = re.search(r'S[0-9]+[wfm]', line)

            # Search for third key patter: Reaktionsblatt or Block followed by a digit
            reaktionsblatt_match = re.search(r'(Reaktionsblatt|Block)\s+(\d+)', line)

            # Search for second key-pattern: a single digit on a line (could've been handeled as an exception and added manually in the data set)
            digit_match = re.match(r'^\s*(\d+)\s*$', line)

            if key_match:
                # Append current body to the corresponding key if already exists
                if key:
                    evaluations.setdefault(key, []).append(body.strip())

                # Set key as S[0-9]+[wfm]
                key = key_match.group()

                # Set Reaktionsblatt as matched group
                reaktionsblatt = reaktionsblatt_match.group(2) if reaktionsblatt_match else reaktionsblatt

                # Append the reaktionsblatt number to the key
                key += f"_{reaktionsblatt}" if reaktionsblatt else ""

                # Prepend the filename without .txt extension to the key
                filename_without_ext = os.path.splitext(os.path.basename(input_file))[0]  # Remove the .txt extension
                key = f"{filename_without_ext}_{key}"

                # Start a new body for this key
                body = line[key_match.end():].strip()

            elif reaktionsblatt_match:
                reaktionsblatt = reaktionsblatt_match.group(2)

            elif digit_match:
                reaktionsblatt = digit_match.group(1)

            elif key:
                # If we're inside a section for a key, continue adding to the body
                body += ' ' + line.strip()

        # Add the final key and body to the dictionary at the end of the file
        if key:
            evaluations.setdefault(key, []).append(body.strip())

    return evaluations

#3.Save into excel. Two methods.

In [ ]:
def dataset_per_sentence_excel (evaluations_dict):

    data_for_excel = []

    for key, evaluations in evaluations_dict.items():
        # Split key: Winter_2023_en_S1f_1 into Semester (Winter_2023), Language (en), Student ID (S1f), and Reaktionsblatt (1)
        parts = key.split("_")
        semester = f"{parts[0]}_{parts[1]}"  # Combine "Winter" and "2023" as "Winter_2023"
        language = parts[2]   # "en"
        student_id = parts[3]  # "S1f"
        reaktionsblatt = parts[4]  # "1"

        # Process each evaluation in the list
        for evaluation in evaluations:
            sentences = re.split(r'(?<=[.!?])\s+', evaluation)
            for sentence_num, sentence in enumerate(sentences, start=1):
                if sentence.strip():  # Avoid empty sentences
                    data_for_excel.append([
                        semester, language, student_id, reaktionsblatt, sentence_num, sentence.strip(), "", "", "", "", "", ""
                    ])  # Empty fields for Labels and Predictions

    columns = [
        'Semester',
        'Language',
        'Student ID',
        'Reaktionsblatt/Block',
        'Sentence Number',
        'Evaluation Text',
        'Label_Positive', 'Label_Neutral', 'Label_Negative',
        'Prediction_Positive', 'Prediction_Neutral', 'Prediction_Negative'
    ]

    df = pd.DataFrame(data_for_excel, columns=columns)

    excel_filename = 'dataset_per_sentence.xlsx'
    df.to_excel(excel_filename, index=False)

    files.download(excel_filename)

In [ ]:
def dataset_per_300words_excel(evaluations_dict):
    # Prepare the data for Excel with expanded structure
    data_for_excel = []

    for key, evaluations in evaluations_dict.items():
        parts = key.split("_")
        semester = f"{parts[0]}_{parts[1]}"
        language = parts[2]
        student_id = parts[3]
        reaktionsblatt = parts[4]

        for evaluation in evaluations:
            # Split evaluation into sentences
            sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', evaluation)
            chunk = []
            chunk_word_count = 0
            chunk_number = 1

            for sentence in sentences:
                sentence_word_count = len(sentence.split())
                # If adding this sentence exceeds 300 words, save the current chunk and start a new one
                if chunk_word_count + sentence_word_count > 300:
                    # Write the current chunk
                    data_for_excel.append([
                        semester, language, student_id, reaktionsblatt, chunk_number, " ".join(chunk).strip(),
                        "", "", "", "", "", ""
                    ])
                    # Start a new chunk
                    chunk = [sentence]
                    chunk_word_count = sentence_word_count
                    chunk_number += 1
                else:
                    # Add the sentence to the current chunk
                    chunk.append(sentence)
                    chunk_word_count += sentence_word_count

            # Write the last chunk (if any sentences remain)
            if chunk:
                data_for_excel.append([
                    semester, language, student_id, reaktionsblatt, chunk_number, " ".join(chunk).strip(),
                    "", "", "", "", "", ""
                ])

    columns = [
        'Semester',
        'Language',
        'Student ID',
        'Reaktionsblatt/Block',
        'Chunk Number',
        'Evaluation Text',
        'Label_Positive', 'Label_Neutral', 'Label_Negative',
        'Prediction_Positive', 'Prediction_Neutral', 'Prediction_Negative'
    ]

    df = pd.DataFrame(data_for_excel, columns=columns)

    excel_filename = 'dataset_per_evaluation.xlsx'
    df.to_excel(excel_filename, index=False)

    files.download(excel_filename)


In [ ]:
from datasets import Dataset

#5.LM

In [ ]:
distilled_student_sentiment_classifier = pipeline(
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    return_all_scores=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
pip install transformers pandas openpyxl

#5.
input:

dataset_per_sentence.xlsx (in method 1)

dataset_per_evaluation.xlsx (in method 2)

In [ ]:
def sentiment_analysis(input_excel):
    """
    Apply sentiment analysis to sentences/parts of evaluations in column F of an Excel file
    and write the sentiment scores into columns J, K, and L of the same file.
    The results will be saved into a renamed Excel file with the prefix 'LM_predictions_'.
    """
    # Read the Excel file
    df = pd.read_excel(input_excel)

    positive_scores = []
    neutral_scores = []
    negative_scores = []

    # Analyze sentiment for each sentence in column F
    for sentence in df['Evaluation Text']:
        if isinstance(sentence, str):  # Ensure the cell contains text
            sentiment_scores = distilled_student_sentiment_classifier(sentence)
            positive_scores.append(sentiment_scores[0][0]['score'])
            neutral_scores.append(sentiment_scores[0][1]['score'])
            negative_scores.append(sentiment_scores[0][2]['score'])
        else:
            # Append default scores for non-text cells
            positive_scores.append(None)
            neutral_scores.append(None)
            negative_scores.append(None)

    # Add the sentiment scores to the DataFrame
    df['Prediction_Positive'] = positive_scores
    df['Prediction_Neutral'] = neutral_scores
    df['Prediction_Negative'] = negative_scores

    # Fill empty fields in columns G, H, and I with zeros
    for column in ['Label_Positive', 'Label_Neutral', 'Label_Negative']:
        if column in df.columns:
            df[column].fillna(0, inplace=True)

    # Generate the new file name with the prefix 'LM_predictions_'
    output_excel = f"LM_predictions_{input_excel}"

    # Save the updated DataFrame to the new Excel file
    df.to_excel(output_excel, index=False)

    print(f"Sentiment analysis results saved to '{output_excel}'.")

In [ ]:
sentiment_analysis("sentence_do_labeling_first_sheet_without_umlauts__no_labels.xlsx")

/tmp/ipython-input-290683887.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[column].fillna(0, inplace=True)


Sentiment analysis results saved to 'LM_predictions_sentence_do_labeling_first_sheet_without_umlauts__no_labels.xlsx'.


In [ ]:
def sentiment_analysis_skip_errors(input_excel):
    """
    Apply sentiment analysis to Evaluation Text in an Excel file.
    If a chunk causes a tensor size error, it is skipped and prediction columns are left empty.
    Saves results to a new Excel file with prefix 'LM_predictions_'.
    """

    df = pd.read_excel(input_excel)

    positive_scores = []
    neutral_scores = []
    negative_scores = []

    for sentence in df['Evaluation Text']:
        if isinstance(sentence, str) and sentence.strip():
            try:
                # Run the model
                sentiment_scores = distilled_student_sentiment_classifier(sentence)
                positive_scores.append(sentiment_scores[0][0]['score'])
                neutral_scores.append(sentiment_scores[0][1]['score'])
                negative_scores.append(sentiment_scores[0][2]['score'])
            except Exception as e:
                # If an error occurs (e.g., tensor size), skip this chunk
                print(f"Skipped a chunk due to error: {e}")
                positive_scores.append(None)
                neutral_scores.append(None)
                negative_scores.append(None)
        else:
            # Non-text or empty cells
            positive_scores.append(None)
            neutral_scores.append(None)
            negative_scores.append(None)

    # Add the prediction columns
    df['Prediction_Positive'] = positive_scores
    df['Prediction_Neutral'] = neutral_scores
    df['Prediction_Negative'] = negative_scores

    # Fill empty label columns (original ones) with zeros if they exist
    for column in ['Label_Positive', 'Label_Neutral', 'Label_Negative']:
        if column in df.columns:
            df[column].fillna(0, inplace=True)

    # Save to new Excel
    output_excel = f"LM_predictions_{input_excel}"
    df.to_excel(output_excel, index=False)

    print(f"Sentiment analysis completed. Results saved to '{output_excel}'")

In [ ]:
sentiment_analysis_skip_errors("sentence_do_labeling_first_sheet_without_umlauts__no_labels.xlsx")

/tmp/ipython-input-890950257.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[column].fillna(0, inplace=True)


Sentiment analysis completed. Results saved to 'LM_predictions_sentence_do_labeling_first_sheet_without_umlauts__no_labels.xlsx'


In [ ]:
from tqdm import tqdm

In [ ]:
def analyze_sentences_with_lm(
    input_file,
    output_csv,
    text_column='Evaluation Text',
    model_name="lxyuan/distilbert-base-multilingual-cased-sentiments-student"
):
    """
    Analyzes each sentence in a CSV or Excel file using a multilingual student sentiment LM
    and saves the results to a UTF-8 CSV file, handling German characters and Excel encoding quirks.

    Parameters:
    - input_file: str, path to input CSV or Excel file
    - output_csv: str, path to save CSV results
    - text_column: str, name of the column containing sentences
    - model_name: str, Hugging Face model name for the LM
    """

    # 1️⃣ Load the file safely
    if input_file.lower().endswith('.csv'):
        try:
            # Try UTF-8 first
            df = pd.read_csv(input_file, encoding='utf-8-sig')
        except UnicodeDecodeError:
            # Fall back to Windows-1252 / cp1252
            df = pd.read_csv(input_file, encoding='cp1252')
    elif input_file.lower().endswith(('.xls', '.xlsx')):
        df = pd.read_excel(input_file, engine='openpyxl')
    else:
        raise ValueError("Unsupported file type. Please provide CSV or Excel.")

    # 2️⃣ Initialize the LM pipeline
    lm_pipeline = pipeline(model_name, return_all_scores=True)

    # 3️⃣ Function to analyze one sentence
    def analyze_sentence(text):
        results = lm_pipeline(str(text))
        scores = {r['label']: r['score'] for r in results[0]}
        majority_label = max(scores.items(), key=lambda x: x[1])[0]
        scores['LM_Label'] = majority_label
        return scores

    # 4️⃣ Apply LM to all sentences with a progress bar
    tqdm.pandas(desc="Analyzing sentences")
    lm_results = df[text_column].progress_apply(analyze_sentence)

    # 5️⃣ Expand results into separate columns
    lm_df = pd.DataFrame(lm_results.tolist())
    df = pd.concat([df, lm_df], axis=1)

    # 6️⃣ Save results to UTF-8 CSV (Excel-safe)
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')

    print(f"✅ LM analysis complete! CSV saved to: {output_csv}")
    return df

In [ ]:
df_results = analyze_sentences_with_lm_csv(
    input_file="FINAL_DATASET_sentence.csv",
    output_csv="FINAL_DATASET_lm_results.csv"
)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x85 in position 1303: invalid start byte

In [ ]:
import pandas as pd
import re

In [ ]:


def normalize_german_umlauts_columnF(input_excel, output_excel):
    """
    Reads an Excel file and replaces German umlauts with ASCII equivalents
    (ä → ae, ö → oe, ü → ue, ß → ss) in column F (6th column),
    then saves the result to a new Excel file.
    """

    # Mapping of German umlauts to ASCII equivalents
    umlaut_map = {
        "ä": "ae",
        "ö": "oe",
        "ü": "ue",
        "Ä": "Ae",
        "Ö": "Oe",
        "Ü": "Ue",
        "ß": "ss"
    }

    # Function to replace umlauts in a string
    def replace_umlauts(text):
        if not isinstance(text, str):
            return text
        for umlaut, replacement in umlaut_map.items():
            text = text.replace(umlaut, replacement)
        return text

    # Read Excel safely
    df = pd.read_excel(input_excel, engine='openpyxl', dtype=str)

    # Get the 6th column (column F) by position
    col_index = 5  # zero-based index, F is 5
    if col_index >= len(df.columns):
        raise ValueError("Excel does not have column F (6th column)")

    col_name = df.columns[col_index]

    # Replace umlauts in that column
    df[col_name] = df[col_name].apply(replace_umlauts)

    # Force all columns to string
    df = df.astype(str)

    # Save Excel safely
    df.to_excel(output_excel, index=False, engine='openpyxl')
    print(f"Normalized Excel saved to {output_excel} with ASCII umlauts in column F.")
    return df


In [ ]:
normalize_german_umlauts("sentence_do_labeling_first_sheet.xlsx", "sentence_do_labeling_first_sheet_without_umlauts.xlsx")

In [ ]:
import pandas as pd
import re


In [ ]:

def restructure_excel_with_chunks(input_excel, output_excel, max_words=270):
    """
    Groups sentence-level rows into evaluation-level chunks.
    Chunk 1 contains as many full sentences as fit into max_words.
    Chunk 2 contains the remaining sentences.
    """

    df = pd.read_excel(input_excel)

    output_rows = []

    group_cols = ['Semester', 'Language', 'Student ID', 'Reaktionsblatt/Block']

    for _, group in df.groupby(group_cols, sort=False):
        group = group.sort_values('Sentence Number')

        chunk_1_sentences = []
        chunk_2_sentences = []

        word_count = 0

        for _, row in group.iterrows():
            sentence = str(row['Evaluation Text']).strip()
            n_words = len(sentence.split())

            if word_count + n_words <= max_words:
                chunk_1_sentences.append(sentence)
                word_count += n_words
            else:
                chunk_2_sentences.append(sentence)

        base_row = group.iloc[0].copy()

        # --- Chunk 1 ---
        base_row_1 = base_row.copy()
        base_row_1['Evaluation Text'] = " ".join(chunk_1_sentences)
        base_row_1['Chunk Number'] = 1
        output_rows.append(base_row_1)

        # --- Chunk 2 ---
        if chunk_2_sentences:
            base_row_2 = base_row.copy()
            base_row_2['Evaluation Text'] = " ".join(chunk_2_sentences)
            base_row_2['Chunk Number'] = 2
            output_rows.append(base_row_2)

    out_df = pd.DataFrame(output_rows)

    # Drop original sentence-level info
    out_df.drop(columns=['Sentence Number',
                         'Label_Positive',
                         'Label_Neutral',
                         'Label_Negative'],
                errors='ignore',
                inplace=True)

    out_df.to_excel(output_excel, index=False)

    print(f"Excel restructured and saved to {output_excel}")


In [ ]:
restructure_excel_with_chunks("sentence_do_labeling_first_sheet_without_umlauts__no_labels.xlsx", "per_evaluation.xlsx")

Excel restructured and saved to per_evaluation.xlsx


In [ ]:


def restructure_excel_with_chunks_2(input_excel, output_excel, max_words=300):
    """
    Groups sentence-level rows from an Excel into evaluation-level chunks.
    Uses the same chunking logic as dataset_per_300words_excel:
    - Sentence-aware
    - Cumulative word count
    - Multiple chunks if necessary
    """

    df = pd.read_excel(input_excel)
    output_rows = []

    group_cols = ['Semester', 'Language', 'Student ID', 'Reaktionsblatt/Block']

    for _, group in df.groupby(group_cols, sort=False):
        group = group.sort_values('Sentence Number')

        # Combine all sentences into a single evaluation string
        evaluation_text = " ".join([str(s).strip() for s in group['Evaluation Text'] if str(s).strip()])

        # Split evaluation into sentences (preserves abbreviations)
        sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', evaluation_text)

        chunk = []
        chunk_word_count = 0
        chunk_number = 1

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue

            sentence_word_count = len(sentence.split())

            # Start new chunk if adding sentence exceeds max_words
            if chunk_word_count + sentence_word_count > max_words and chunk:
                # Save current chunk
                base_row = group.iloc[0].copy()
                base_row['Evaluation Text'] = " ".join(chunk).strip()
                base_row['Chunk Number'] = chunk_number
                output_rows.append(base_row)

                # Start new chunk
                chunk = [sentence]
                chunk_word_count = sentence_word_count
                chunk_number += 1
            else:
                # Add sentence to current chunk
                chunk.append(sentence)
                chunk_word_count += sentence_word_count

        # Save the last chunk
        if chunk:
            base_row = group.iloc[0].copy()
            base_row['Evaluation Text'] = " ".join(chunk).strip()
            base_row['Chunk Number'] = chunk_number
            output_rows.append(base_row)

    out_df = pd.DataFrame(output_rows)

    # Drop columns no longer needed
    out_df.drop(columns=['Sentence Number',
                         'Label_Positive',
                         'Label_Neutral',
                         'Label_Negative'],
                errors='ignore',
                inplace=True)

    # Save the restructured Excel
    out_df.to_excel(output_excel, index=False)

    print(f"Excel restructured and saved to {output_excel}")


In [ ]:
restructure_excel_with_chunks_2("sentence_do_labeling_first_sheet_without_umlauts__no_labels.xlsx", "per_evaluation.xlsx")

Excel restructured and saved to per_evaluation.xlsx


In [ ]:
def averaged_scores_sentence(input_excel, output_excel):
    """
    Computes the average values of columns G to L for each evaluation (grouped by columns A to D)
    and writes the results into a new Excel file, keeping all original column names.
    """

    # Read the Excel file
    df = pd.read_excel(input_excel)

    # Define group and value columns
    group_cols = ['Semester', 'Language', 'Student ID', 'Reaktionsblatt/Block']
    value_cols = [
        'Label_Positive', 'Label_Neutral', 'Label_Negative',
        'Prediction_Positive', 'Prediction_Neutral', 'Prediction_Negative'
    ]

    # Check for required columns
    missing_cols = [col for col in group_cols + value_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    # Ensure numeric values
    df[value_cols] = df[value_cols].apply(pd.to_numeric, errors='coerce')

    # Sum and count for each group
    grouped_sum = df.groupby(group_cols)[value_cols].sum()
    sentence_counts = df.groupby(group_cols).size()

    # Compute averages
    averaged_df = grouped_sum.div(sentence_counts, axis=0).reset_index()

    # Keep original column names
    averaged_df = averaged_df[group_cols + value_cols]

    # Save the result
    averaged_df.to_excel(output_excel, index=False)
    print(f"✅ Averages computed and saved to '{output_excel}'")

In [ ]:
averaged_scores_sentence("LM_predictions_sentence_do_labeling_first_sheet_without_umlauts__no_labels.xlsx", "averaged_LM_labelled_sentence.xlsx")

✅ Averages computed and saved to 'averaged_LM_labelled_sentence.xlsx'
